In [13]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [46]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('gsk_') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama-3.3-70b-versatile"'
openai = OpenAI()

API key looks good so far


In [47]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [48]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [49]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [50]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [63]:
client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

In [67]:
def select_relevant_links(url):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [68]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'main website', 'url': 'https://edwarddonner.com/'},
  {'type': 'blog/posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [71]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [72]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama-3.3-70b-versatile"
Found 7 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog/posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'personal website', 'url': 'https://edwarddonner.com/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'company website',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [73]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile"
Found 8 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [74]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    for link in relevant_links["links"]:
        print("Opening:", link["url"])  # Shows which URL is being opened

        result += f"\n\n### Link: {link['type']}\n"

        try:
            result += fetch_website_contents(link["url"])
        except Exception as e:
            result += f"\nError fetching {link['url']}: {e}\n"

    return result




In [75]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile"
Found 6 relevant links
Opening: https://huggingface.co
Opening: https://huggingface.co/blog
Opening: https://apply.workable.com/huggingface/
Opening: https://huggingface.co/learn
Opening: https://huggingface.co/enterprise
Opening: https://huggingface.co/docs
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
empero-ai/Qwythos-9B-Claude-Mythos-5

In [76]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [77]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [78]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile"
Found 6 relevant links
Opening: https://huggingface.co/blog
Opening: https://apply.workable.com/huggingface/
Opening: https://huggingface.co
Opening: https://huggingface.co/docs
Opening: https://discuss.huggingface.co
Opening: https://huggingface.co/changelog


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nempero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF\nUpdated\n10 days ago\n•\n1.68M\n•\n1.83k\ntencent/Hy3\nUpdated\n2 days

In [82]:
def create_brochure(company_name, url):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [83]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile"
Found 5 relevant links
Opening: https://huggingface.co/blog
Opening: https://apply.workable.com/huggingface/
Opening: https://huggingface.co/learn
Opening: https://huggingface.co/docs
Opening: https://huggingface.co/enterprise


# Introduction to Hugging Face
Hugging Face is a cutting-edge AI community that is building the future of machine learning. The company provides a platform where the machine learning community can collaborate on models, datasets, and applications.

## Our Mission
At Hugging Face, our mission is to create, discover, and collaborate on machine learning better. We aim to provide a platform that fosters innovation, creativity, and progress in the field of AI.

## Company Culture
Our company culture is built around the principles of collaboration, openness, and innovation. We believe in the power of the community and strive to create an environment where everyone can contribute, learn, and grow. Our community is at the heart of everything we do, and we are committed to supporting and empowering our users to achieve their goals.

## Our Customers
Our platform is used by a wide range of customers, from individual researchers and developers to large enterprises. We provide a vast array of models, datasets, and applications that can be used in various industries, including but not limited to:
* Natural Language Processing (NLP)
* Computer Vision (CV)
* Reinforcement Learning (RL)
* Audio

## Careers at Hugging Face
We are always looking for talented individuals to join our team. Our careers page lists all the current openings, and we encourage you to check it out if you are interested in working with us. We offer a dynamic and collaborative work environment, with opportunities for professional growth and development.

## Learning and Development
At Hugging Face, we are committed to helping our users learn and develop their skills in machine learning. Our learn page provides access to various courses, tutorials, and resources, including:
* LLM Course: Learn about large language models using libraries from the HF ecosystem
* Context Course: Learn context engineering for code agents
* Robotics Course: Build robots with LeRobot
* Smol Course: A smallest course on machine learning

## Get Involved
If you are interested in getting involved with our community, you can start by exploring our platform, browsing our models, datasets, and applications, and joining our community forum. We also have a blog where we post articles, research papers, and news about the latest developments in AI and machine learning.

## Join Us
Whether you are a researcher, developer, or simply someone interested in AI and machine learning, we invite you to join our community and be a part of shaping the future of AI. Sign up for our platform, follow us on social media, and stay up-to-date with the latest news and developments from Hugging Face.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [87]:
def stream_brochure(company_name, url):
    stream = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [88]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile"
Found 5 relevant links
Opening: https://huggingface.co
Opening: https://huggingface.co/blog
Opening: https://apply.workable.com/huggingface/
Opening: https://huggingface.co/docs
Opening: https://discuss.huggingface.co


# Hugging Face Brochure
## Introduction
Hugging Face is a cutting-edge AI company that serves as a collaboration platform for the machine learning community. The platform enables users to create, discover, and collaborate on models, datasets, and applications. With a strong focus on open-source collaboration, Hugging Face is dedicated to building the future of AI.

## Our Mission
At Hugging Face, our mission is to provide a platform where the machine learning community can come together to collaborate, share knowledge, and drive innovation. We strive to make AI more accessible and usable for everyone, and to promote a culture of open-source collaboration and transparency.

## Company Culture
Our company culture is centered around collaboration, innovation, and community. We believe in the power of open-source collaboration and strive to create an environment that fosters creativity, inclusivity, and transparency. Our team is passionate about AI and dedicated to making a positive impact on the world.

## Customers
Our platform is used by a wide range of customers, from individual researchers and developers to large enterprises. Our customers use our platform to build and deploy AI models, collaborate on research projects, and develop innovative applications. We serve customers across various industries, including technology, healthcare, finance, and education.

## Careers
We are always looking for talented and passionate individuals to join our team. If you are interested in AI, machine learning, or software development, we encourage you to check out our career page. We offer a range of roles, from engineering and research to marketing and sales. Our team is dedicated to making a positive impact on the world, and we are committed to creating a work environment that is inclusive, supportive, and fun.

## Products and Services
Our platform offers a range of products and services, including:
* **Models**: We provide access to over 2 million pre-trained models, including state-of-the-art models for natural language processing, computer vision, and more.
* **Datasets**: Our platform hosts over 500,000 datasets, making it easy for users to find and access the data they need for their projects.
* **Spaces**: We offer a range of pre-built applications and tools, including voice chat, image editing, and video generation.
* **Buckets**: Our platform provides secure and scalable storage for models, datasets, and applications.
* **Enterprise Solutions**: We offer customized solutions for large enterprises, including enterprise support, inference providers, and storage buckets.

## Community
Our community is at the heart of everything we do. We believe in the power of collaboration and strive to create a platform that is inclusive, supportive, and fun. Our community includes thousands of researchers, developers, and industry professionals who are passionate about AI and machine learning. We offer a range of community features, including forums, blogs, and social media channels.

## Join Us
If you are interested in learning more about Hugging Face, we encourage you to explore our website, join our community, or check out our career page. We are always looking for talented and passionate individuals to join our team and help us build the future of AI.

In [89]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile"
Found 5 relevant links
Opening: https://huggingface.co
Opening: https://huggingface.co/blog
Opening: https://apply.workable.com/huggingface/
Opening: https://huggingface.co/docs
Opening: https://huggingface.co/learn


# Hugging Face Brochure
## Introduction
Hugging Face is a cutting-edge AI community that is building the future of machine learning. As a collaboration platform, Hugging Face enables the machine learning community to come together and work on models, datasets, and applications.

## Our Mission
At Hugging Face, our mission is to provide a platform where the machine learning community can collaborate, create, and discover new AI models and applications. We strive to move the field of machine learning forward by providing a space for innovation and experimentation.

## Products and Services
Hugging Face offers a range of products and services, including:
* **Models**: Browse over 2 million pre-trained models and create your own
* **Datasets**: Access over 500,000 datasets and contribute your own
* **Spaces**: Run and deploy AI applications with ease
* **Buckets**: Store and manage your data with our cloud storage solution
* **Hugging Face PRO**: Get access to premium features and support for your business
* **Inference Providers**: Leverage our network of inference providers for fast and reliable deployment

## Community
At Hugging Face, we believe that community is at the heart of everything we do. Our community is made up of researchers, developers, and industry leaders who are passionate about machine learning. Join our community to:
* **Collaborate** on projects and research
* **Share** your knowledge and expertise
* **Learn** from others and stay up-to-date with the latest developments in AI

## Careers
We're always looking for talented individuals to join our team. If you're passionate about machine learning and want to work with a dynamic and innovative company, check out our **careers page** for available positions.

## Culture
At Hugging Face, we value:
* **Openness**: We believe in open-source collaboration and transparency
* **Innovation**: We encourage experimentation and creativity
* **Community**: We're dedicated to building a strong and supportive community
* **Ethics**: We prioritize responsible AI development and use

## Get Involved
Ready to join the Hugging Face community? Sign up for an account, explore our platform, and start contributing to the future of machine learning. Follow us on social media and join the conversation on our **Discord** and **Forum** channels.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>